# MRKR contralateral TKA - Phase 2 training on Colab

Runs **`src/train_model.py`**, the device-agnostic trainer, on a Colab GPU.

This notebook does **not** replace `notebooks/train_colab.ipynb`, which stays as the record of the original Colab path. It exists because only `src/train_model.py` writes the hand-over artefacts that `src/eval_models.py` and `src/manuscript_figures.py` consume: `derived-data/cohort/val_hazards_{arm}.npz` and `derived-data/cohort/train_arms.json`.

`resolve_device` is `cuda -> mps -> cpu` and AMP is gated on `model_image.local.amp_devices: ["cuda"]`, so CUDA and AMP are picked up here with **no code change and no config edit**.

**The locked test split is never read.** The sealed path is not implemented in this trainer at all. `outputs/crop_qa_checklist.md` stays unsigned and still blocks the test read later.

### Non-negotiables
1. **Never downscale the 512x512 crop** (protocol section 13). If memory is tight, raise `--grad-accum`. This is what `notebooks/train_colab.ipynb` cell 34 says to do.
2. **The flags are load-bearing.** `grad_accum_steps`, `micro_batch_size` and `num_workers` are inside `TrainSettings.training_contract()` and therefore inside the contract hash. `train_one_seed` **refuses** a checkpoint written under a different hash. Once you pick flags in step 5, every later invocation and every resume must repeat them exactly.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM {p.total_memory/1024**3:.1f} GB")
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU, then rerun.")

## 2. Mount Drive and check the upload

Expected Drive layout (create it before running):

```
MyDrive/mrkr/
    mrkr_colab_code.tar.gz      <- the 368 KB code bundle
    shards/
        train-00000.tar         <- 372.4 MB
        val-00000.tar           <-  52.2 MB
        labels.csv              <-   1.1 MB
    ckpt/                       <- created here; checkpoints are mirrored into it
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE   = pathlib.Path('/content/drive/MyDrive/mrkr')
BUNDLE  = DRIVE / 'mrkr_colab_code.tar.gz'
DSHARDS = DRIVE / 'shards'
DCKPT   = DRIVE / 'ckpt'
DCKPT.mkdir(parents=True, exist_ok=True)

for p in (BUNDLE, DSHARDS / 'train-00000.tar', DSHARDS / 'val-00000.tar',
          DSHARDS / 'labels.csv'):
    assert p.exists(), f"missing on Drive: {p}"
    print(f"ok  {p.name:22s} {p.stat().st_size/1048576:8.1f} MB")

## 3. Unpack code, stage shards on local SSD

The shards are **copied off Drive to local disk**. Drive FUSE is slow and the trainer reads every shard once to build its cache; paying that over FUSE wastes minutes.

`${HOME}` on Colab is `/root`, and `src/config.py` resolves paths with `os.path.expandvars`, so `${HOME}/mrkr-shards` becomes `/root/mrkr-shards` on its own. **No config edit is needed.**

In [ ]:
import os, shutil, sys, tarfile, time

PROJ = pathlib.Path('/content/mrkr-project')
if PROJ.exists():
    shutil.rmtree(PROJ)
with tarfile.open(BUNDLE) as tf:
    tf.extractall('/content')
shutil.move('/content/project', str(PROJ))

SH = pathlib.Path('/root/mrkr-shards')
SH.mkdir(parents=True, exist_ok=True)
t0 = time.time()
for n in ('train-00000.tar', 'val-00000.tar', 'labels.csv'):
    if not (SH / n).exists():
        shutil.copy2(DSHARDS / n, SH / n)
print(f"shards staged in {time.time()-t0:.0f}s")

os.chdir(PROJ)
sys.path.insert(0, str(PROJ))
print(pathlib.Path.cwd())
!ls -la /root/mrkr-shards

## 4. Restore previous checkpoints, then install dependencies

Checkpoints live on **local disk** (`/root/mrkr-ckpt`) because the trainer rewrites a 461 MB file every epoch and doing that over Drive FUSE would dominate the epoch time. They are **mirrored to Drive** by the background sync in step 6 and restored here.

In [ ]:
CK = pathlib.Path('/root/mrkr-ckpt')
CK.mkdir(parents=True, exist_ok=True)
restored = 0
for f in sorted(DCKPT.glob('*.pt')):
    dst = CK / f.name
    if not dst.exists() or dst.stat().st_size != f.stat().st_size:
        shutil.copy2(f, dst)
        restored += 1
print(f"restored {restored} checkpoint(s) from Drive; {len(list(CK.glob('*.pt')))} present")

In [ ]:
# Colab already ships a CUDA-built torch/torchvision. Do NOT `pip install -r
# requirements-training.txt`: its torch==2.13.0 pin would replace Colab's CUDA build
# with one that may not match the driver.
!pip -q install timm lifelines duckdb 2>&1 | tail -2

import importlib
for m in ("timm", "lifelines", "duckdb", "patsy", "statsmodels",
          "pyarrow", "sklearn", "yaml", "PIL"):
    try:
        mod = importlib.import_module(m)
        print(f"  {m:14s} {getattr(mod, '__version__', 'ok')}")
    except ImportError as e:
        print(f"  {m:14s} MISSING -> {e}")

## 5. Smoke test, then calibrate `--grad-accum` on this GPU

The smoke test does one forward/backward plus the numpy-vs-torch NLL check (must agree to 1e-6) and asserts the image path: attention sums to 1.0 with zero mass on padded slots, and the 31 px border is still exactly zero after the affine.

Then `--time-steps` measures real step time, so the accumulation choice is a measurement rather than a guess.

**Sizing.** ConvNeXt-Tiny holds roughly 0.5 GB of fp32 backward activations per 512x512 crop, halved under AMP. Batching is per **patient** and crops per patient vary (mean 1.64, max 5), so a micro-batch of 8 patients can carry up to 23 crops. Start from this table, then confirm with the timing.

| GPU | VRAM | start with |
|---|---|---|
| T4 | 16 GB | `--grad-accum 8` (4 patients per micro-batch) |
| L4 | 24 GB | `--grad-accum 4` |
| A100 | 40 GB | `--grad-accum 4` |

For reference, on a 24 GB M4 Pro at `--grad-accum 8` the measured cost was 334 s/epoch for `m4_fusion` on a healthy machine.

In [ ]:
# --smoke defaults to m4_fusion so the image-path assertions actually run.
!python -m src.train_model --smoke

In [ ]:
GRAD_ACCUM  = 4      # set from the table above, then confirm with the timing below
NUM_WORKERS = 2      # Colab VMs are small; 2 is usually right

!python -m src.train_model --time-steps 20 --grad-accum {GRAD_ACCUM} --num-workers {NUM_WORKERS}

## 6. Mirror checkpoints to Drive in the background

Colab disconnects. This copies new or updated checkpoints to Drive every 5 minutes, so a timeout costs at most the in-flight epoch. Step 4 restores them on the next session.

In [ ]:
import subprocess, textwrap

sync = textwrap.dedent(f'''
    import shutil, time, pathlib
    src = pathlib.Path("/root/mrkr-ckpt")
    dst = pathlib.Path("{DCKPT}")
    while True:
        try:
            for f in sorted(src.glob("*.pt")):
                d = dst / f.name
                if (not d.exists()) or d.stat().st_size != f.stat().st_size \\
                        or d.stat().st_mtime < f.stat().st_mtime:
                    shutil.copy2(f, d)
        except Exception as e:
            print("sync warn:", e, flush=True)
        time.sleep(300)
''')
pathlib.Path('/content/ckpt_sync.py').write_text(sync)
SYNC = subprocess.Popen(['python', '/content/ckpt_sync.py'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
print("checkpoint sync running, pid", SYNC.pid)

## 7. Stage 1 - the manuscript-critical arms

`m0d_clinical`, `m1_klg`, `m4_fusion` (the primary model), `m3_image`. Ordered so the two clinical arms finish in minutes and the manuscript is unblocked early.

Safe to re-run after a disconnect: finished seeds are skipped and an interrupted seed resumes from its last epoch. **The flags must match step 5 exactly.**

In [ ]:
!python -m src.train_model --stage stage1 --grad-accum {GRAD_ACCUM} --num-workers {NUM_WORKERS}

## 8. Stage 2 - protocol completeness

`m2_frontal`, `m4_frontal` (section 24 view comparison), `r1_densenet_frontal` (section 25 robustness, trained after the primary model is frozen).

In [ ]:
!python -m src.train_model --stage stage2 --grad-accum {GRAD_ACCUM} --num-workers {NUM_WORKERS}

## 9. Collect the artefacts to bring home

These are everything `src/eval_models.py` needs. Download `mrkr_trained_artifacts.tar.gz` and unpack it over the local project root.

The npz files carry per-patient validation hazards keyed by `empi_anon`, so they are patient-level: they belong in `derived-data/`, which is git-ignored, and must never be committed.

In [ ]:
OUTT = pathlib.Path('/content/mrkr_trained_artifacts.tar.gz')
want = (sorted(pathlib.Path('derived-data/cohort').glob('val_hazards_*.npz'))
        + [pathlib.Path('derived-data/cohort/train_arms.json'),
           pathlib.Path('outputs/tables/train_history.csv'),
           pathlib.Path('outputs/tables/seed_variability.csv')])
missing = [str(p) for p in want if not p.exists()]
assert not missing, f"not produced: {missing}"

with tarfile.open(OUTT, 'w:gz') as tf:
    for p in want:
        tf.add(p, arcname=str(p))
        print(f"  {str(p):52s} {p.stat().st_size/1024:8.1f} KB")

shutil.copy2(OUTT, DRIVE / OUTT.name)
print(f"\nwrote {OUTT} ({OUTT.stat().st_size/1048576:.1f} MB) and copied to Drive")

In [ ]:
import pandas as pd
print(pd.read_csv('outputs/tables/seed_variability.csv').to_string(index=False))

## Resuming after a disconnect

Run cells **1, 2, 3, 4, 6**, then whichever of **7 / 8** had not finished. Keep `GRAD_ACCUM` and `NUM_WORKERS` identical to the first run (cell 5), or every existing checkpoint is refused for a foreign contract hash.

If you must change them, every checkpoint is invalidated: rerun with `--force-retrain` and record it in `outputs/protocol_deviations.md`.